# Preprocessing Imagery

## **Overview:**

1. Load scene manifest
2. Cloud masking with S2 SCL (for L2A only) + Clip to AOI
3. 2D verification plot
4. Save processed scenes and the catalog of masked and clipped scenes.

**Memory Strategy**: Cloud masking and clipping will be executed together by loading and processing scene by scene.

---

## Background and Prerequisites
Background on S2 SCL, and the projects area of interst (AOI) Upper Keys, FL, are necessary for this notebook.

### **S2 Scene Classification Layer (SCL)**
20 m resolution classification layer based on visibility of satelliete imagery. 

| Class | Meaning |
| -- | --- |
| 0 | No Data |
| 1 | Saturated/Defective |
| 2 | Dark area pixels |
| 3 | Cloud Shadows |
| 4 | Vegetation |
| 5 | Bare soils |
| 6 | Water |
| 7 | Cloud low probability |
| 8 | Cloud medium probability |
| 9 | Cloud high probability |
| 10 | Thin cirrus |
| 11 | Snow |

Cloud masking commonly filters out classes 3, 7, 8, 9, and 10. The SCL only applys to class L2A and therefore excludes the years 2015-2016 in this analysis.

#### AOI: Upper Keys FL (John Pennekamp Coral Reef State Park)
Shape file of the AOI provided by the Florida Fish and Wildlife Conservation Commission via the [Unified Florida Reef Tract Map](https://geodata.myfwc.com/documents/myfwc::unified-florida-reef-tract-map/about). Export of feature John Pennekamp Coral Reef State Park as `UpperKeys_FL.shp` was done in ArcGIS Pro using select by attribute and export feature selection.
<div style="display: flex; justify-content: left;"> 
    <img src="https://www.arcgis.com/sharing/rest/content/items/6f4c1a90afc84222945874345fb092a2/info/thumbnail/thumbnail1701868505714.png?w=800" width="400" height="200"> 
    <div>


---

## Imports

In [ ]:
# pip install...

In [ ]:
import sys
import os
import re
import gc
import glob
import xarray as xr
import rioxarray as rxr
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

### Paths and Directories


In [ ]:
manifest_path = "processed/scene_manifest.csv"
sentinel_dir = "C:/Users/sulli/DATA_Science/seagrass_project/sentinel_data"
aoi_shapefile = "C:/Users/sulli/DATA_Science/seagrass_project/study_area/UpperKeys_FL.shp"
clipped_dir = "processed/clipped"

# SCL cloud classes to mask (3=cloud shadow, 7=unclassified, 8=cloud medium, 9=cloud high, 10=thin cirrus)
cloud_classes = [3, 7, 8, 9, 10]

os.makedirs(clipped_dir, exist_ok=True)

## 1. Load Manifest and Study Area

In [ ]:
# Load study scene manifest and count scenes by product level
df = pd.read_csv("processed/scene_manifest.csv")
print(df["level"].value_counts())
# ensure 2015-2016 is L1C
print(df[df["year"] <= 2016][["scene_id", "year", "level"]].head(10))


In [ ]:
manifest = pd.read_csv(manifest_path, parse_dates=["date"])
print(f"Scenes in manifest: {len(manifest)}")
print(f"L1C: {(manifest['level']=='L1C').sum()}     L2A: {(manifest['level']=='L2A').sum()}")
display(manifest.head(2))
display(manifest.tail(2))

In [ ]:
# Load study area shapefile
aoi = gpd.read_file(aoi_shapefile)

# print CRS
print(f"AOI CRS: {aoi.crs}")

# Simple plot of study area polygon
aoi.plot()
plt.title("Study Area - Upper Keys FL")
plt.show()

## Function: Find SCL File
SCL layer uses 20m resolution band inside `.SAFE` folder and is only available for L2A (2017-2024 for this analysis).

In [ ]:
def find_scl(scene_id, year, sentinel_dir):
    """Locate the SCL jp2file for a given scene_id inside sentinel2_data."""
    year_path = os.path.join(sentinel_dir, str(year))

    # match SAFE folder containing the scene datetime
    datetime_str = scene_id.split("_")[1]
    safe_matches = glob.glob(os.path.join(year_path, f"*{datetime_str}*.SAFE"))
    if not safe_matches:
        return None
        
    safe = safe_matches[0]
    scl_files = glob.glob(os.path.join(safe, "**/*SCL_20m.jp2"), recursive=True)

    return scl_files[0] if scl_files else None

## Cloud Mask & Clip 
The main loop for each scene: 
- opens `.nc` DataSet from `01_access_data` 
- reprojects AOI to match CRS of scenes
    -   **L2A only:** find SCL in `SAFE` and match 20m band resolution to 10m bands with `reproject_match`, and apply cloud mask. 
    -   **L1C:** skipping cloud masking
- Clip all bands to AOI
- Save as `processed/clipped/{scene_id}_clipped.nc`, delete from memory.

Note: L1C is only clipped, this most likely will not affect results as cloud coverage was less than 30% when searching.

In [ ]:
clipped_records = []
saved_count = 0
skipped_count = 0
failed_count = 0

# Iterate through scene manifest 
for _, row in manifest.iterrows():
    scene_id = row["scene_id"]
    year     = row["year"]
    level    = row["level"]
    nc_path  = row["nc_path"]
    out_path = os.path.join(clipped_dir, f"{scene_id}_clipped.nc")

    # Skipping if exists
    if os.path.exists(out_path):
        skipped_count += 1
        clipped_records.append({"scene_id": scene_id,
                                "level": level,
                                "year": year,
                                "date": row["date"],
                                "clipped_path": out_path
        })
        continue
    if not os.path.exists(nc_path):
        print(f"    SKIP (nc not found): {scene_id}")
        failed_count += 1 
        continue

    try:
        # open netCDF scene DataSet
        ds = xr.open_dataset(nc_path)
        # Ensure CRS is defined 
        ds = ds.rio.write_crs(ds.rio.crs or "EPSG:32617")

        # reproject AOI to match scene CRS
        aoi_proj = aoi.to_crs(ds.rio.crs)

        # STEP 1: cloud masking for L2A only
        if level == "L2A":
            scl_path = find_scl(scene_id, year, sentinel_dir)
            if scl_path:
                # Load SCL and rematch 20m resolution to 10m bands
                scl = rxr.open_rasterio(scl_path, masked=True).squeeze("band", drop=True)
                scl_10m = scl.rio.reproject_match(ds["blue"]).squeeze("band", drop=True)

                # build cloudmask: True where cloudy
                cloud_mask = np.isin(scl_10m.values, cloud_classes)

                # apply to all bands
                for var in ds.data_vars:
                    ds[var] = ds[var].where(~cloud_mask)

                del scl, scl_10m, cloud_mask
            else:
                print(f"    WARNING (no SCL found, skipping cloud mask): {scene_id}")
        # STEP 2: Clip to AOI
        ds = ds.rio.clip(aoi_proj.geometry, aoi_proj.crs, drop=True, invert=False)

        # STEP 3: Memory-first strategy:
        # immediately save using 'zlib' complevel=4 to balance speed and size
        encoding = {v: {"zlib": True, "complevel": 4} for v in ds.data_vars}
        ds.to_netcdf(out_path, encoding=encoding)

        clipped_records.append({
                                "scene_id": scene_id,
                                "level": level,
                                "year": year,
                                "date": row["date"],
                                "clipped_path": out_path
                                })
        saved_count += 1

    except Exception as e:
        print(f"    ERROR ({scene_id}): {e}")
        failed_count += 1

    finally:
        # Cleanup: Close ds and force gc.collect() to clean RAM
        try:
            ds.close()
        except:
            pass
        gc.collect()

print(f"\nDone. saved: {saved_count}    skipped: {skipped_count}    failed: {failed_count}")

In [ ]:
print(f"Scenes with SCL warnings: check clipped_manifest for flag")
# quick check - how many clipped files exist
import glob
clipped = glob.glob("processed/clipped/*.nc")
print(f"Clipped files on disk: {len(clipped)}")

## Save Clipped Manifest 

In [ ]:
# Sort clipped_records with year and scene ID by creating a Pandas DataFrame
clipped_df = (
    pd.DataFrame(clipped_records)
    .sort_values(["year", "scene_id"])
    .reset_index(drop=True)
)

# Save preprocessed clipped_manifest of netCDF scenes
clipped_df.to_csv("processed/clipped_manifest.csv", index=False)
print(f"Clipped manifest saved. Total: {len(clipped_df)}")
clipped_df.head()

## Verification Plot
A 2D plot of one clipped scene (NIR band) to confirm the CRS alignment, clip and cloud mask application is correct, using `plt.subplots()` and `boundary.plot()` to overlay the study area boundary.


In [ ]:
# pick the first clipped scene
sample = clipped_df.iloc[0]
ds_plot = xr.open_dataset(sample["clipped_path"])

aoi_plot = aoi.to_crs(ds_plot.rio.crs)

fig, ax = plt.subplots(figsize=(8,8))

# plot NIR band
ds_plot["nir"].plot(
    ax=ax,
    cmap="Greens",
    robust=True,
    add_colorbar=True,
)

# overlay study area boundary using `boundary.plot()`
aoi_plot.boundary.plot(ax=ax, color="red", linewidth=1.5, label="Study Area")

ax.set_title(f"{sample['scene_id']} - NIR clipped ({sample['level']})")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.legend()
plt.tight_layout()
plt.show()

ds_plot.close()

---

## Summary
**Outputs:**
- `processed/clipped/{scene_id}_clipped.nc` - Clipped and cloud masked, for L2A, scenes.
- `processed/clipped_manifest.csv` - metadata summary of procesesd scenes

**Next:**
`03_calculate_indices` to load clipped manifest and water quality data from `02b_preprocess_data`, compute NDVI, EVI, NDWI and other vegetation indices 

## Resources and references-
 - Elshall, A. (2026). *xarray* [Environmental Data Science course notebook] FGCU.
 - Code assistance provided by Kiro AI (AWS, 2025). https://kiro.dev
 - Wu, Qiusheng (2024). [Rioxarray.](https://geog-312.gishub.org/book/geospatial/rioxarray.html) In GIS Programming. 
 - Wasser, L., et. al. (2022, Jan 21). [*Overlay Raster and Vector Spatial Data in A Matplotlib Plot Using Extents in Python.*](https://earthdatascience.org/courses/scientists-guide-to-plotting-data-in-python/plot-spatial-data/customize-raster-plots/plotting-extents/) Earth Data Science.  
 